## Exp-metrics analysis

In [1]:
import datetime
import pandas as pd

from expmetrics.datasets.segment.flags_first_user_exposures import FlagsFirstUserExposures
from exp_analysis_tmp.main import ExpAnalysisRunner

In [2]:

def filter_results(results: pd.DataFrame, segment_key: str | None = None, metric_name: str | None = None) -> pd.DataFrame:
    """Filter results by segment key and metric name."""
    if segment_key:
        results = results[results['segment_key'] == segment_key]
    if metric_name:
        results = results[results['metric_name'] == metric_name]
    return results

First, we need to define flag key and date range for the experiment we want to analyze (start and end date).

In [3]:
EXPERIMENT_KEY = "sca_payments_ab_test_v2"
EXPERIMENT_START_DATE = datetime.datetime(2025, 11, 25, 0, 0, 0, 0, tzinfo=datetime.timezone.utc)
COMPUTATION_DATE = datetime.datetime(2025, 12, 31, 0, 0, 0, 0, tzinfo=datetime.timezone.utc)

As at the moment variant names are not standardized in feature-flag experiments, we need to define the control variant name manually.

Here, you can also define which metrics you want to analyze. **IMPORTANT**: If a metric has lag value larger than 0, the compute_to date will be adjusted by subtracting the lag value from it. If the compute_to date is before the lag value, the computation will not be performed.

In [4]:
CONTROL_VARIANT_NAME = "control_group"

METRICS = [
    #payouts
    "buyers_exposed_time_filter",
    "gmv_purchased_exposed_time_filter",
    "buyers",
    "last_payment_attempts_success",
    "last_payments_success_rate",
    "all_payment_attempts_success",
    "all_payments_success_rate"



]

As svc-abs experiment-based metric computations used specific segment datasets, built around either A/B test exposures or A/B test assignments, you have to change those with a flag-based exposure or assignment datasets.

While defining the `SEGMENT_DATASETS_MAP`, define which segment dataset should replace the existing dataset. Have in mind that while computing metrics, tables are joined through entity key, so for example, computing transactions for VintedGo lockers is not possible. In addition you should define the `kwargs_to_inherit` dictionary, which will be used to inherit the existing dataset's parameters (for example suspicious users filter, etc.).

In [5]:
SEGMENT_DATASETS_MAP = {
    "AbTestExpose": {
        "alternative_class": FlagsFirstUserExposures,
        "kwargs_to_inherit": {
            "user_filter_type": "_user_filter_type",
        },
    }
}

Then all you need to do is to intialize analysis runner and run the analysis:

In [7]:
exp_analysis_runner = ExpAnalysisRunner(
    segment_dataset_map=SEGMENT_DATASETS_MAP,
    control_variant_name=CONTROL_VARIANT_NAME,
)

Here metric results will automatically be computed per experiment level and per all of the additional segments defined within the segment dataset definition:

In [ ]:
results = exp_analysis_runner.run(
    experiment_key=EXPERIMENT_KEY,
    compute_from=EXPERIMENT_START_DATE,
    compute_to=COMPUTATION_DATE,
    metrics=METRICS,
)

View specific metric results for overall (experiment-level) segment:

In [9]:
# Create a summary table with metric name, target average, control average, group counts, and p-value
# Filter to OVERALL segment to get experiment-level results
overall_results = filter_results(results, segment_key="OVERALL")

# Create summary table
summary_table = overall_results[[
    'metric_name', 
    'control_avg', 
    'variant_avg', 
    'control_cnt',
    'variant_cnt',
    'p_val'
]].copy()

# Rename columns for clarity
summary_table = summary_table.rename(columns={
    'metric_name': 'Metric Name',
    'control_avg': 'Control Average',
    'variant_avg': 'Target Average',
    'control_cnt': 'Control Group Size',
    'variant_cnt': 'Target Group Size',
    'p_val': 'P Value'
})

# Format averages to 3 decimal places
summary_table['Control Average'] = summary_table['Control Average'].apply(
    lambda x: f"{x:.3f}" if pd.notna(x) else "N/A"
)
summary_table['Target Average'] = summary_table['Target Average'].apply(
    lambda x: f"{x:.3f}" if pd.notna(x) else "N/A"
)

# Format group sizes as integers with thousand separators
summary_table['Control Group Size'] = summary_table['Control Group Size'].apply(
    lambda x: f"{int(x):,}" if pd.notna(x) else "N/A"
)
summary_table['Target Group Size'] = summary_table['Target Group Size'].apply(
    lambda x: f"{int(x):,}" if pd.notna(x) else "N/A"
)

# Format p-value with more decimal places
summary_table['P Value'] = summary_table['P Value'].apply(
    lambda x: f"{x:.6f}" if pd.notna(x) else "N/A"
)

# Sort by metric name
summary_table = summary_table.sort_values('Metric Name').reset_index(drop=True)

# Display the table
summary_table

,Metric Name,Control Average,Target Average,Control Group Size,Target Group Size,P Value
0,all_payment_attempts_success,0.943,0.849,"1,050","1,110",0.000000
1,buyers,0.998,0.980,"1,050","1,113",0.000074
2,buyers_exposed_time_filter,0.976,0.958,"1,050","1,113",0.017003
3,gmv_purchased_exposed_time_filter,1430.843,1368.351,"1,050","1,113",0.396278
4,last_payment_attempts_success,0.985,0.965,"1,050","1,110",0.000387


View specific metric results for country level segment:

In [10]:
# Create a summary table with differences and color coding
overall_results = filter_results(results, segment_key="OVERALL")

# Create summary table
summary_table = overall_results[[
    'metric_name', 
    'control_avg', 
    'variant_avg', 
    'rel_diff',
    'control_cnt',
    'variant_cnt',
    'abs_diff',
    'p_val',
]].copy()

# Store raw values BEFORE formatting for styling
summary_table['_rel_diff_raw'] = summary_table['rel_diff']
summary_table['_p_val_raw'] = summary_table['p_val']

# Rename columns
summary_table = summary_table.rename(columns={
    'metric_name': 'Metric Name',
    'control_avg': 'Control Average',
    'variant_avg': 'Target Average',
    'abs_diff': 'Absolute Difference',
    'rel_diff': 'Relative Difference (%)',
    'p_val': 'P Value',
    'control_cnt': 'Control Group Size',
    'variant_cnt': 'Target Group Size',
})

# Format values
summary_table['Control Average'] = summary_table['Control Average'].apply(
    lambda x: f"{x:.3f}" if pd.notna(x) else "N/A"
)
summary_table['Target Average'] = summary_table['Target Average'].apply(
    lambda x: f"{x:.3f}" if pd.notna(x) else "N/A"
)
summary_table['Absolute Difference'] = summary_table['Absolute Difference'].apply(
    lambda x: f"{x:.3f}" if pd.notna(x) else "N/A"
)
summary_table['Relative Difference (%)'] = summary_table['Relative Difference (%)'].apply(
    lambda x: f"{x*100:.2f}%" if pd.notna(x) else "N/A"
)
summary_table['P Value'] = summary_table['P Value'].apply(
    lambda x: f"{x:.3f}" if pd.notna(x) else "N/A"
)
summary_table['Control Group Size'] = summary_table['Control Group Size'].apply(
    lambda x: f"{int(x):,}" if pd.notna(x) else "N/A"
)
summary_table['Target Group Size'] = summary_table['Target Group Size'].apply(
    lambda x: f"{int(x):,}" if pd.notna(x) else "N/A"
)


# Style: mark red if relative difference is negative and significant (p < 0.05)
def style_row(row):
    """Style relative difference column"""
    rel_diff = row['_rel_diff_raw']
    p_val = row['_p_val_raw']
    
    styles = [''] * len(row)
    rel_diff_idx = row.index.get_loc('Relative Difference (%)')
    
    if pd.notna(rel_diff) and pd.notna(p_val):
        if abs(rel_diff) > 0 and p_val < 0.05:
            styles[rel_diff_idx] = 'background-color: #ffcccc'
    
    return styles

# Apply styling
styled = summary_table.style.apply(style_row, axis=1)

print("="*80)
print("PAYINS SCA TEST V2")
print("="*80)

styled

PAYINS SCA TEST V2


,Metric Name,Control Average,Target Average,Relative Difference (%),Control Group Size,Target Group Size,Absolute Difference,P Value,_rel_diff_raw,_p_val_raw
0,buyers_exposed_time_filter,0.976,0.958,-1.89%,"1,050","1,113",-0.018,0.017,-0.018868,0.017003
1,gmv_purchased_exposed_time_filter,1430.843,1368.351,-4.37%,"1,050","1,113",-62.492,0.396,-0.043675,0.396278
2,buyers,0.998,0.980,-1.79%,"1,050","1,113",-0.018,0.000,-0.017896,0.000074
157,last_payment_attempts_success,0.985,0.965,-1.99%,"1,050","1,110",-0.020,0.000,-0.019912,0.000387
185,all_payment_attempts_success,0.943,0.849,-9.95%,"1,050","1,110",-0.094,0.000,-0.099530,0.000000


In [ ]:
#

## Further BQ analysis

In [11]:
import pydata_google_auth
credentials = pydata_google_auth.get_user_credentials(
    [
        'https://www.googleapis.com/auth/cloud-platform', 
        'https://www.googleapis.com/auth/bigquery'
    ],
    use_local_webserver=False
)

import pandas_gbq as pdq

# It should output your vinted email
pdq.read_gbq('select SESSION_USER()').head()
%load_ext google.cloud.bigquery

Downloading: 100%|██████████|


/Users/gabijastaponaite/Documents/GitHub/untitled folder/exp-metrics/.venv/lib/python3.12/site-packages/google/cloud/bigquery/__init__.py:239: FutureWarning: %load_ext google.cloud.bigquery is deprecated. Install bigquery-magics package and use `%load_ext bigquery_magics`, instead.
  warnings.warn(


In [12]:
%%bigquery dt --use_rest_api --maximum_bytes_billed=100000000000

-- Step 1: Users exposed to sca_payouts_ab_test
WITH test_users AS (
  SELECT DISTINCT
    CAST(targeting_key AS INT64) AS user_id,
    variant
  FROM `vi-dv-prod-grf.contracts.backend_flags_exposure`
  WHERE time >= '2025-11-11 10:30:00'
    AND flag_key = 'sca_payouts_ab_test_v2'
),

-- Step 2: Payout-exposed users who are in the A/B test
exposed_users AS (
  SELECT
    a.user_id AS seller_id,
    status_code,
    amount_eur,
    failure_reason,
    payout_id,
    tu.variant
  FROM `dv-mkp-pmnt-prod.marts.mrt_payouts` a
  INNER JOIN `dv-mkp-pmnt-prod.dimensions.dim_accounts` ac
    ON a.user_id = ac.user_id
  INNER JOIN test_users tu
    ON a.user_id = tu.user_id
  WHERE a.created_at >= '2025-11-11 10:30:00'
    AND amount_eur > 500
)

-- Step 3: Most popular failure reasons by variant
SELECT
  variant,
  failure_reason,
  COUNT(DISTINCT payout_id) AS payout_count,
  COUNT(DISTINCT seller_id) AS seller_count,
  SUM(amount_eur) AS total_amount_eur
FROM exposed_users
WHERE failure_reason IS NOT NULL
GROUP BY variant, failure_reason
ORDER BY variant, payout_count DESC

Query is running:   0%|          |

Downloading:   0%|          |

In [13]:
dt

,variant,failure_reason,payout_count,seller_count,total_amount_eur
0,control_group,User blocked by Fraud policy,1405,275,1.584508e+06
1,control_group,Transaction blocked by Fraud Policy,256,83,2.501187e+05
2,control_group,Insufficient wallet balance,145,29,2.387957e+05
3,control_group,Blocked due to the Bank Account User's KYC lim...,138,109,9.547225e+04
4,control_group,MS03,138,92,1.812323e+05
...,...,...,...,...,...
233,target_group,Account holder with id AH322VL223222B5G2SSWLBG...,1,1,1.421296e+03
234,target_group,Account holder with id AH329SF22322945MKQTPG9P...,1,1,2.773669e+03
235,target_group,[ORGNFPID/010F27425363D8FU 10202] Rejection C...,1,1,6.171212e+02
236,target_group,[ORGNFPID/010F27425322499O 10202] Rejection C...,1,1,3.281941e+03
